# Prueba de ejecución

In [1]:
library("duckdb")

Loading required package: DBI



In [2]:
# to start an in-memory database
con <- dbConnect(duckdb())

duckdb is storing downloaded extensions and secrets under ~/.duckdb:
ℹ /home/milo/.duckdb
This persists across sessions and is shared with the DuckDB CLI and other clients.
ℹ Run duckdb(shared_home = FALSE) to use a temporary directory instead.
ℹ See ?duckdb_storage for details and alternatives.


In [3]:
dbExecute(con, "
INSTALL iceberg;
LOAD iceberg;

CREATE SECRET iceberg_secret (
             TYPE iceberg,
             CLIENT_ID 'spark',
             CLIENT_SECRET '<CLIENT_SECRET>',
             OAUTH2_SERVER_URI '<OAUTH2_SERVER_URI>',
             OAUTH2_SCOPE 'lakekeeper'     
	);
	
LOAD httpfs;
ATTACH 'cepcan' AS cepcan (
   TYPE iceberg,
   SECRET iceberg_secret, -- pass a specific secret name to prevent ambiguity
   ENDPOINT '<ENDPOINT>'
);
")

[1] 0

In [4]:
# retrieve the items again
res <- dbGetQuery(con, "SELECT * FROM cepcan.pyiceberg_namespace.cdata;")
res

Municipio,Actividad económica,UE Unidades económicas,Año Censal,diversity,ubiquity,mcp,eci,pci,density,coi,cog,rca,Entidad,A111A Producción bruta total (millones de pesos)
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
001 Abalá,Rama 1125 Acuicultura,0,2023,21,472,0,-0.6648572,-0.60170476,0.127474888,-0.7209931,0.328752607,0.000000,NA,NA
001 Abalá,Rama 1141 Pesca,0,2023,21,304,0,-0.6648572,-0.52837021,0.106811988,-0.7209931,0.523824749,0.000000,NA,NA
001 Abalá,Rama 1151 Servicios relacionados con la agricultura,0,2023,21,269,0,-0.6648572,0.92439446,0.074308453,-0.7209931,1.101944610,0.000000,NA,NA
001 Abalá,Rama 1152 Servicios relacionados con la cría y explotación de animales,0,2023,21,117,0,-0.6648572,1.74990870,0.044232434,-0.7209931,1.647409030,0.000000,NA,NA
001 Abalá,Rama 1153 Servicios relacionados con el aprovechamiento forestal,0,2023,21,21,0,-0.6648572,3.39789276,0.026401363,-0.7209931,1.151537125,0.000000,NA,NA
001 Abalá,Rama 2111 Extracción de petróleo y gas,0,2023,21,17,0,-0.6648572,4.00458246,0.020216325,-0.7209931,1.638400397,0.000000,NA,NA
001 Abalá,Rama 2121 Minería de carbón mineral,0,2023,21,8,0,-0.6648572,2.68273908,0.023385461,-0.7209931,0.922778154,0.000000,NA,NA
001 Abalá,Rama 2122 Minería de minerales metálicos,0,2023,21,103,0,-0.6648572,0.66982184,0.047952789,-0.7209931,1.280370497,0.000000,NA,NA
001 Abalá,Rama 2123 Minería de minerales no metálicos,0,2023,21,381,0,-0.6648572,-0.06693827,0.103133185,-0.7209931,0.568487238,0.000000,NA,NA


In [ ]:
# Writing to Iceberg
dbExecute(con, "
CREATE SCHEMA cepcan.sales;
USE cepcan.sales;

CREATE TABLE cepcan.sales.events (
    id INTEGER,
    event_name VARCHAR,
    event_time TIMESTAMP
);
")

In [6]:
# Table Properties
## Existing properties can be inspected and modified with the property functions:
dbExecute(con, "
 -- View properties
SELECT * FROM iceberg_table_properties(cepcan.sales.events);   
")

[1] 0

In [7]:
# Inserting Data
dbExecute(con, "
INSERT INTO cepcan.sales.events
VALUES (1, 'click', TIMESTAMP '2026-06-01 10:00:00');
")

[1] 1

In [8]:
#
dbGetQuery(con, "SELECT * FROM cepcan.sales.events;")

id,event_name,event_time
<int>,<chr>,<dttm>
1,click,2026-06-01 10:00:00
1,click,2026-06-01 10:00:00
